# Workbook 2: Data wrangling with pandas (banking data)

Most real data-science time goes on cleaning and reshaping. You'll use `drop_duplicates`, `to_datetime`, `groupby`, `transform`, `pivot_table` and `pd.cut`.

*Builds on:* TS Module 1: pandas reading, filtering, aggregation, missing values

**How to use this workbook**
1. Run the setup cell below first (`Shift + Enter`).
2. For each exercise, replace `raise NotImplementedError` with your code.
3. Run the `check(...)` cell. ✅ means done; ❌ tells you which check failed.
4. Stuck? Follow the stuck routine in `PRACTICE.md` before you look at `exercises/solutions.py`.

Try things out in a scratch cell (`Insert → Cell Below`): print values, test small pieces, make mistakes.

In [1]:
import sys
from pathlib import Path

# Find the Finance-ds folder so this notebook can use the repo's helper code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fintools").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from exercises.check import check           # check(your_function) grades your answer

## 2.1 `clean_transactions`

Return a cleaned copy of a raw transactions DataFrame:

In [7]:
def clean_transactions(df):
    """Return a cleaned copy of a raw transactions DataFrame:
    - drop duplicate rows by tx_id (keep first)
    - convert `timestamp` to datetime
    - fill missing `amount` with the median amount of that row's `category`
    - sort by timestamp and reset the index
    Do not modify the input DataFrame.
    """
    df = df.copy()
    df = df.drop_duplicates(subset="tx_id", keep="first")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["amount"] = df.groupby("category")["amount"].transform(
        lambda s: s.fillna(s.median())
    )
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

In [8]:
check(clean_transactions)

✅ clean_transactions: all checks passed!


True

## 2.2 `monthly_spend`

Pivot table: one row per customer_id, one column per month (pd.Period 'M'),

In [11]:
def monthly_spend(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])   # ← make sure it's datetime
    df["month"] = df["timestamp"].dt.to_period("M")
    return df.pivot_table(
        index="customer_id",
        columns="month",
        values="amount",
        aggfunc="sum",
        fill_value=0,
    )

In [12]:
check(monthly_spend)

✅ monthly_spend: all checks passed!


True

## 2.3 `top_customers`

Series of the n customers with the highest total spend, largest first (index = customer_id).

In [14]:
def top_customers(df, n=5):
    """Series of the n customers with the highest total spend, largest first (index = customer_id)."""
    totals = df.groupby("customer_id")["amount"].sum()
    totals = totals.sort_values(ascending=False)
    return totals.head(n)

In [15]:
check(top_customers)

✅ top_customers: all checks passed!


True

## 2.4 `default_rate_by_band`

Default rate per FICO-style credit band.

In [21]:
def default_rate_by_band(loans, bins=(300, 580, 670, 740, 800, 851)):
    """Default rate per FICO-style credit band.
    Bands (left-inclusive): Poor [300,580), Fair [580,670), Good [670,740),
    Very good [740,800), Excellent [800,851). Return a Series indexed by band label.
    Hint: pd.cut(..., right=False, labels=[...]) then groupby.
    """
    labels = ["Poor", "Fair", "Good", "Very good", "Excellent"]
    band = pd.cut(loans["credit_score"], bins=bins, right=False, labels=labels)
    return loans.groupby(band, observed=False)["default"].mean()

In [22]:
check(default_rate_by_band)

✅ default_rate_by_band: all checks passed!


True

## ✍️ Reflect (write in your learning log)

- Which exercise was hardest, and what was the concept you were missing?
- Pick one function and explain in 2–3 plain sentences what it does **for the business**.
- Did you peek at a solution? Note it, and redo that exercise from memory next session.


Worked through data-cleaning and customer-analysis exercises: cleaning
transactions, monthly spend grid, top customers, and default rate by
credit band.

**What clicked:**
The pattern groupby → sum/mean → sort → head. And reading each docstring
as a manager's question ("who are our top customers?"), not a puzzle.

**What was hard / where I got stuck:**
- monthly_spend crashed because timestamp was text, not a date. Fixed
  by adding pd.to_datetime at the start of the function.
- default_rate_by_band failed with KeyError 'Very good'. Still need to
  figure out why some band labels go missing.

**My answer to a question I'm proud of:**
top_customers answers "who are our biggest spenders?" — it adds up each
customer's spending and lists the top 5, biggest first. Useful for
rewarding loyal customers or spotting concentration risk.

**One thing I can do now that I couldn't before:**
Group data and summarise it — e.g. total spend per customer, default
rate per segment — instead of just looking at raw rows.